In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

#import numpy as np # linear algebra
#import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import time

# --- 1. Environment Setup (Handling dependencies) ---
print("Installing dependencies...")
# Use existing local wheels in the dataset
os.system("pip install --no-index --no-deps /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl")
os.system("pip install --no-index --no-deps /kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl")

# --- 2. Configuration & Paths ---
DATA_BASE = Path("/kaggle/input/competitions/stanford-rna-3d-folding-2")
TEST_CSV = DATA_BASE / "test_sequences.csv"
OUTPUT_PATH = "submission.csv"

# --- 3. Refinement Logic for Better Scores ---
def apply_physical_constraints(coords):
    """
    Enforce RNA physical constraints to improve TM-score.
    RNA C1'-C1' distance is typically ~5.9 Angstroms.
    """
    refined = coords.copy()
    # Simple smoothing to prevent impossible jumps
    if len(refined) > 1:
        for i in range(1, len(refined)):
            dist = np.linalg.norm(refined[i] - refined[i-1])
            if dist > 10.0: # If distance is physically impossible
                direction = (refined[i] - refined[i-1]) / dist
                refined[i] = refined[i-1] + direction * 5.9
    return refined

# --- 4. Main Inference Function ---
def run_inference():
    print("Loading test sequences...")
    if not TEST_CSV.exists():
        print(f"Error: {TEST_CSV} not found.")
        return

    test_df = pd.read_csv(TEST_CSV)
    
    # --- ERROR FIX: Handle column names dynamically ---
    # The competition uses 'target_id', but some notebooks use 'id'
    id_col = 'target_id' if 'target_id' in test_df.columns else 'id'
    seq_col = 'sequence'
    
    submission_data = []
    print(f"Processing {len(test_df)} sequences using column: {id_col}")

    for idx, row in test_df.iterrows():
        target_id = row[id_col]
        sequence = row[seq_col]
        seq_len = len(sequence)
        
        # --- Placeholder for Model Prediction ---
        # Replace this part with your Protenix/Model call to get (N_SAMPLES, seq_len, 3)
        # Here we create a dummy structure that follows a basic helix for demo
        # coords_5_models shape: (5, seq_len, 3)
        coords_5_models = []
        for m in range(5):
            # Generate dummy coordinates (replace with your model.predict)
            dummy_coords = np.zeros((seq_len, 3))
            for i in range(seq_len):
                angle = i * 0.5 + m
                dummy_coords[i] = [10 * np.cos(angle), 10 * np.sin(angle), i * 3.0]
            
            # Apply refinement to each model
            refined_coords = apply_physical_constraints(dummy_coords)
            coords_5_models.append(refined_coords)
        
        # --- 5. Format to Competition Submission Format ---
        # Format: ID (target_id_resid), resname, resid, x_1, y_1, z_1 ... x_5, y_5, z_5
        for i in range(seq_len):
            res_name = sequence[i]
            res_id = i + 1
            row_id = f"{target_id}_{res_id}"
            
            line = [row_id, res_name, res_id]
            for m in range(5):
                line.extend(coords_5_models[m][i]) # Add x, y, z for model m
            
            submission_data.append(line)

    # Create DataFrame
    cols = ['ID', 'resname', 'resid']
    for m in range(1, 6):
        cols += [f'x_{m}', f'y_{m}', f'z_{m}']
    
    sub_df = pd.DataFrame(submission_data, columns=cols)
    sub_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Successfully saved submission to {OUTPUT_PATH}")

if __name__ == "__main__":
    start_time = time.time()
    try:
        run_inference()
        print(f"Total time: {time.time() - start_time:.2f} seconds")
    except Exception as e:
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()

Installing dependencies...
Processing /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
biopython is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
biotite is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Loading test sequences...
Processing 28 sequences using column: target_id
Successfully saved submission to submission.csv
Total time: 0.68 seconds
